# 6. HTTP Headers

**Headers** are extra information sent with an API request or response.

The URL says where the request is going. The method says what action we want. Headers add details like:

- what kind of data we can accept
- what kind of data we are sending
- which app is making the request
- whether we have permission to access the API
- how many requests we have left before a rate limit


In [1]:
import requests
from pprint import pprint

TIMEOUT = 10

## Example 1: Call a Real API With Request Headers

GitHub has a public API that works without authentication for basic requests.

We will send two useful headers:

- `Accept`: asks for GitHub's JSON response format
- `User-Agent`: identifies the program making the request


In [2]:
github_headers = {
    "Accept": "application/vnd.github+json",
    "User-Agent": "Intro-API-Class/1.0",
}

response = requests.get(
    "https://api.github.com/repos/pallets/flask",
    headers=github_headers,
    timeout=TIMEOUT,
)

print(response.status_code)
print(response.url)

200
https://api.github.com/repos/pallets/flask


The body is still JSON, so we can read it the same way we did in earlier lessons.


In [3]:
repo = response.json()

print(repo["full_name"])
print(repo["description"])
print("Stars:", repo["stargazers_count"])
print("Language:", repo["language"])

pallets/flask
The Python micro framework for building web applications.
Stars: 71685
Language: Python


## Example 2: Inspect Response Headers

The API response includes headers too. GitHub sends useful headers about data format and rate limits.


In [4]:
print("Content-Type:", response.headers.get("Content-Type"))
print("Rate limit:", response.headers.get("X-RateLimit-Limit"))
print("Rate limit remaining:", response.headers.get("X-RateLimit-Remaining"))
print("Rate limit resets at:", response.headers.get("X-RateLimit-Reset"))

Content-Type: application/json; charset=utf-8
Rate limit: 60
Rate limit remaining: 52
Rate limit resets at: 1781675232


Headers are dictionary-like. We can loop through the ones we care about.


In [5]:
headers_to_check = [
    "Date",
    "Server",
    "Content-Type",
    "Cache-Control",
    "X-GitHub-Api-Version-Selected",
]

for header_name in headers_to_check:
    print(header_name, "->", response.headers.get(header_name))

Date -> Wed, 17 Jun 2026 05:23:23 GMT
Server -> github.com
Content-Type -> application/json; charset=utf-8
Cache-Control -> public, max-age=60, s-maxage=60
X-GitHub-Api-Version-Selected -> 2022-11-28


## Example 3: Inspect the Headers Python Sent

`response.request.headers` shows the headers that Python sent to the API.


In [6]:
print(response.request.method)
print(response.request.url)

for header_name in ["Accept", "User-Agent"]:
    print(header_name, "->", response.request.headers.get(header_name))

GET
https://api.github.com/repos/pallets/flask
Accept -> application/vnd.github+json
User-Agent -> Intro-API-Class/1.0


## Example 4: Use an API That Echoes Headers Back

httpbin is a public testing API. Its `/headers` endpoint sends our request headers back in the response body.

This is helpful when we want to prove that our custom headers were actually sent.


In [7]:
class_headers = {
    "User-Agent": "Intro-API-Class/1.0",
    "Accept": "application/json",
    "X-Class-Topic": "headers",
    "X-Student-Mode": "practice",
}

headers_response = requests.get(
    "https://httpbin.org/headers",
    headers=class_headers,
    timeout=TIMEOUT,
)

print(headers_response.status_code)
pprint(headers_response.json()["headers"])

200
{'Accept': 'application/json',
 'Accept-Encoding': 'gzip, deflate, br',
 'Host': 'httpbin.org',
 'User-Agent': 'Intro-API-Class/1.0',
 'X-Amzn-Trace-Id': 'Root=1-6a322f62-011a30f51321adeb342d8f3e',
 'X-Class-Topic': 'headers',
 'X-Student-Mode': 'practice'}


Headers that start with `X-` are custom headers. They are not built into HTTP, but APIs sometimes use them for extra metadata.


In [8]:
echoed_headers = headers_response.json()["headers"]

print("Class topic:", echoed_headers.get("X-Class-Topic"))
print("Student mode:", echoed_headers.get("X-Student-Mode"))

Class topic: headers
Student mode: practice


## Example 5: `Content-Type` With a JSON `POST`

When we use `json=payload`, `requests` sends JSON and automatically sets `Content-Type` to `application/json`.


In [9]:
payload = {
    "title": "Headers lesson",
    "body": "requests can set Content-Type automatically",
    "userId": 1,
}

post_response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=payload,
    timeout=TIMEOUT,
)

print(post_response.status_code)
print("Sent Content-Type:", post_response.request.headers.get("Content-Type"))
pprint(post_response.json())

201
Sent Content-Type: application/json
{'body': 'requests can set Content-Type automatically',
 'id': 101,
 'title': 'Headers lesson',
 'userId': 1}


## Example 6: Compare JSON Data and Form Data

`json=` and `data=` both send information, but they use different `Content-Type` headers.


In [10]:
form_payload = {
    "name": "Ada",
    "favorite_topic": "headers",
}

json_response = requests.post(
    "https://httpbin.org/post",
    json=form_payload,
    timeout=TIMEOUT,
)

form_response = requests.post(
    "https://httpbin.org/post",
    data=form_payload,
    timeout=TIMEOUT,
)

print("json= Content-Type:", json_response.request.headers.get("Content-Type"))
print("data= Content-Type:", form_response.request.headers.get("Content-Type"))

json= Content-Type: application/json
data= Content-Type: application/x-www-form-urlencoded


In [11]:
json_data = json_response.json()
form_data = form_response.json()

print("Data received from json= request:")
pprint(json_data["json"])

print()
print("Data received from data= request:")
pprint(form_data["form"])

Data received from json= request:
{'favorite_topic': 'headers', 'name': 'Ada'}

Data received from data= request:
{'favorite_topic': 'headers', 'name': 'Ada'}


## Example 7: A Small Helper Function

A helper function can keep API header setup in one place.


In [12]:
def get_github_repo(owner, repo_name):
    url = f"https://api.github.com/repos/{owner}/{repo_name}"
    headers = {
        "Accept": "application/vnd.github+json",
        "User-Agent": "Intro-API-Class/1.0",
    }

    response = requests.get(url, headers=headers, timeout=TIMEOUT)
    response.raise_for_status()
    return response

python_response = get_github_repo("python", "cpython")
python_repo = python_response.json()

print(python_repo["full_name"])
print("Open issues:", python_repo["open_issues_count"])
print("Rate limit remaining:", python_response.headers.get("X-RateLimit-Remaining"))

python/cpython
Open issues: 9300
Rate limit remaining: 51


## Quick Review

- Request headers are sent from Python to the API.
- Response headers are sent from the API back to Python.
- `Accept` tells the API what kind of response we want.
- `Content-Type` tells the API what kind of data we are sending.
- `User-Agent` identifies our client.
- `Authorization` is commonly used for API keys, tokens, and other credentials. We will handle real API keys in the next lesson.
